# 01 Split Protocol Validation

This notebook validates the time-based train/validation/test protocol and checks for leakage risks.

In [1]:
from pathlib import Path
import pandas as pd
import sys

SCRIPTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts')
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from preprocessing import CLEAN_DIR, split_summary

In [2]:
def load_clean(dataset: str) -> pd.DataFrame:
    df = pd.read_csv(CLEAN_DIR / f'dataset_{dataset.lower()}_clean.csv')
    df['month'] = pd.to_datetime(df['month'])
    return df

In [3]:
checks = []
for dataset in ['A', 'B', 'C']:
    df = load_clean(dataset)
    split_df = split_summary(df)
    span = split_df.groupby('split')['month'].agg(['min', 'max', 'nunique']).reset_index()
    display(dataset)
    display(span)
    checks.append({
        'dataset': dataset,
        'train_rows': int((split_df['split'] == 'train').sum()),
        'val_rows': int((split_df['split'] == 'val').sum()),
        'test_rows': int((split_df['split'] == 'test').sum()),
        'n_series': int(split_df['series_id'].nunique()),
    })
pd.DataFrame(checks)

'A'

,split,min,max,nunique
0,test,2025-02-01,2026-01-01,12
1,train,2023-02-01,2024-07-01,18
2,val,2024-08-01,2025-01-01,6


'B'

,split,min,max,nunique
0,test,2025-02-01,2026-01-01,12
1,train,2023-02-01,2024-07-01,18
2,val,2024-08-01,2025-01-01,6


'C'

,split,min,max,nunique
0,test,2025-02-01,2026-01-01,12
1,train,2023-02-01,2024-07-01,18
2,val,2024-08-01,2025-01-01,6


,dataset,train_rows,val_rows,test_rows,n_series
0,A,1854,618,1236,103
1,B,1854,618,1236,103
2,C,111240,37080,74160,6180


## Dataset C Scenario Split Validation

In [4]:
c = load_clean('C')
c[['scenario_id','scenario_split']].drop_duplicates()['scenario_split'].value_counts()

scenario_split
train    45
test     15
Name: count, dtype: int64

In [5]:
c.groupby('scenario_split')['scenario_id'].nunique()

scenario_split
test     15
train    45
Name: scenario_id, dtype: int64

## Leakage Checks

In [6]:
for dataset in ['A','B','C']:
    df = load_clean(dataset)
    split_df = split_summary(df)
    leakage = split_df.duplicated(subset=['series_id','month']).sum()
    print(dataset, 'duplicated series/month pairs in split summary =', int(leakage))